# Section 1: Introduction

## Contribution of this Notebook
This notebook serves as an empirical bridge between the theoretical literature of Large Language Models (LLMs) and their real-world behaviors. By synthesizing the findings of the 2026 *Large Language Models Hallucination: A Comprehensive Survey*, this work transitions abstract concepts into executable code. It demonstrates hallucination mechanisms empirically, proving that theoretical flaws in autoregressive modeling actively manifest as factual fabrications during text generation.

## What is Hallucination in Large Language Models?
Hallucination refers to the generation of content by an LLM that is fluent and syntactically correct but factually inaccurate or completely unsupported by external evidence. 

> [!IMPORTANT]
> **Key Insight:** Hallucination is an emergent property of probabilistic language modeling.

In the context of probabilistic language modeling, LLMs do not "know" facts; they estimate the probability of the next token given a sequence of preceding tokens. When the statistical likelihood of a token sequence aligns with human knowledge, we perceive it as "factual." When it diverges—due to bias, missing data, or architectural limits—we perceive it as a hallucination.

**Types of Hallucination:**
* **Intrinsic:** Contradicts the source material.
* **Extrinsic:** Adds unverifiable information not present in the source.
* **Factual:** Contradicts real-world facts.
* **Faithfulness:** Disobeys user instructions or logical consistency.

# Section 2: Theoretical Foundations

## Maximum Likelihood Estimation (MLE)
Most LLMs are trained using Maximum Likelihood Estimation (MLE). The objective is to maximize the probability of the correct next token $y_t$ given the context $x$:
$$ \mathcal{L}_{MLE} = - \sum_{t=1}^T \log P(y_t | y_{<t}, x) $$
While MLE strongly encourages syntactic fluency and adherence to the training distribution, it does not penalize factual contradictions unless those contradictions explicitly lower the likelihood. **LLMs optimize probability, not truth.**

## Exposure Bias
During pre-training, the model is fed perfect, ground-truth tokens (a process known as Teacher Forcing). However, during inference, it generates tokens autoregressively based on its *own* past predictions. If it makes an early mistake, the error snowballs because the model has never learned to recover from its own flawed context.

## Softmax Bottleneck & Temperature
The final layer of an LLM uses a Softmax function to produce a probability distribution over the entire vocabulary. 
$$ P(y_i) = \frac{\exp(z_i / T)}{\sum_j \exp(z_j / T)} $$
Where $T$ is the temperature. The Softmax bottleneck occurs when multiple words are equally valid; the math struggles to represent them all fairly, sometimes forcing the model into a statistically weak, hallucinated path. High temperature flattens the distribution, increasing randomness and the probability of sampling "creative" but hallucinated tokens.

## Entropy and Uncertainty
Entropy measures the uncertainty in the model's prediction:
$$ H = -\sum P(x) \log P(x) $$
A high entropy value indicates that the probability distribution is flat—the model is "unsure" of the next token and is essentially guessing. However, a low entropy value (high confidence) does not guarantee truthfulness; it only means the model strongly believes a token fits the statistical pattern.

# Section 3: Theoretical Limitations

## Why Hallucination is Inevitable

Hallucination cannot be entirely eliminated because it is fundamentally wired into the architecture of modern LLMs. 

* **Mismatch between Likelihood and Truth:** LLMs are statistical pattern matchers, not databases. They optimize for the most probable continuation of text, not the most factually correct one.
* **Autoregressive Limitations:** The left-to-right generation mechanism means the model cannot easily backtrack or revise a mistaken premise once it has committed a token to the sequence.
* **Trade-off between Fluency and Factuality:** The exact same probabilistic mechanism that allows an LLM to generate creative poetry, brainstorm novel ideas, and understand nuanced human language is what causes it to hallucinate facts. If you constrain an LLM to be perfectly deterministic, it loses its conversational fluency.

# Section 4: LLM Hallucination Pipeline

To understand where hallucinations originate, we must examine the entire development pipeline. Below is a conceptual diagram of the vulnerability points in an LLM system:

```mermaid
graph TD
    A[Data Curation] -->|Bias & Imitative Falsehoods| B(Pre-Training: MLE)
    B -->|Exposure Bias & Shortcut Learning| C{Model Architecture}
    C -->|Soft Attention Decay & Softmax Bottleneck| D(Fine-Tuning / RLHF)
    D -->|Capability Misalignment & Sycophancy| E[Inference / Decoding]
    E -->|Temperature Randomness| F((Hallucinated Output))
    
    style A fill:#f9d0c4,stroke:#333,stroke-width:2px
    style B fill:#f9d0c4,stroke:#333,stroke-width:2px
    style C fill:#f9d0c4,stroke:#333,stroke-width:2px
    style D fill:#f9d0c4,stroke:#333,stroke-width:2px
    style E fill:#f9d0c4,stroke:#333,stroke-width:2px
    style F fill:#e74c3c,color:#fff,stroke:#333,stroke-width:4px
```

### Pipeline Vulnerability Points:
1. **Data Curation:** If the training data contains societal myths, the model learns "imitative falsehoods."
2. **Pre-Training:** Teacher forcing creates "exposure bias," leaving the model unable to self-correct during inference.
3. **Model Architecture:** Soft attention decays over long sequences, and the Softmax bottleneck struggles to represent equally probable truths.
4. **Fine-Tuning (RLHF):** Humans reward confident-sounding answers. This alignment trains the model into "sycophancy," where it prefers to lie confidently rather than admit ignorance.
5. **Inference (Decoding):** High temperature introduces stochastic randomness, directly correlating with extrinsic hallucinations.

# Section 5: Experimental Setup
We will use HuggingFace's `transformers` library with a lightweight model (`gpt2`) to demonstrate these phenomena. The principles shown here apply identically to larger state-of-the-art models.

In [ ]:
import torch
import math
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch.nn.functional as F

model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
# Ensure padding token is set
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(model_id)

def generate_text(prompt, max_new_tokens=30, temperature=1.0, do_sample=True, return_probs=False):
    inputs = tokenizer(prompt, return_tensors="pt", padding=True)
    outputs = model.generate(
        inputs['input_ids'],
        attention_mask=inputs['attention_mask'],
        max_new_tokens=max_new_tokens, 
        temperature=temperature, 
        do_sample=do_sample,
        pad_token_id=tokenizer.eos_token_id,
        return_dict_in_generate=True,
        output_scores=True
    )
    
    text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    
    if return_probs:
        return text, outputs.scores, outputs.sequences[0]
    return text

# Section 6: Experiment 1 — Hallucination Examples
### Explanation
We prompt the model with an ambiguous or entirely fictional query to observe confident hallucinations. By feeding the model a fabricated historical entity, we test its reliance on linguistic interpolation over factual databases.

In [ ]:
prompt_fictional = "The famous 19th-century Martian explorer, Dr. Elias Thorne, is best known for his discovery of"
output_fictional = generate_text(prompt_fictional, temperature=0.7, max_new_tokens=20)

print("Prompt:", prompt_fictional)
print("-" * 30)
print("Output:", output_fictional)

### Interpretation
The model confidently hallucinates a continuation for a completely fabricated entity. Because it is optimizing for linguistic likelihood, it acts as an interpolator, failing to distinguish between historical fact and a plausible syntactic continuation.

# Section 7: Experiment 2 — Decoding Effects (Temperature)
### Explanation
We analyze how varying the temperature affects the likelihood of hallucination by testing deterministic vs. highly stochastic generation.

In [ ]:
prompt_factual = "The capital of France is"

print("Low Temperature (0.1) - Highly Deterministic:")
print(generate_text(prompt_factual, temperature=0.1, do_sample=True, max_new_tokens=5))

print("\nHigh Temperature (2.0) - Highly Random (Prone to Extrinsic Hallucination):")
print(generate_text(prompt_factual, temperature=2.0, do_sample=True, max_new_tokens=15))

### Interpretation
Lower temperatures sharpen the softmax distribution, yielding factual and repetitive answers. Higher temperatures flatten the distribution, increasing the chance of sampling low-probability, irrelevant, or factually incorrect tokens.

# Section 8: Experiment 3 — Uncertainty Analysis
### Explanation
We inspect the token probabilities and compute the Shannon entropy of the distribution. 
* **High entropy** = The model is highly uncertain (distribution is flat).
* **Low entropy** = The model is confident (distribution is peaked).

> [!WARNING]
> **Key Insight:** Confidence does not imply correctness in LLM outputs. A model can be highly confident about a hallucinated token.


In [ ]:
prompt_uncertain = "The inventor of the quantum neural network in 1985 was"
text, scores, sequences = generate_text(prompt_uncertain, temperature=1.0, return_probs=True, max_new_tokens=5)

print("Generated Text:", text)
print("-" * 50)
print("Token-level Uncertainty Analysis:\n")

input_length = len(tokenizer(prompt_uncertain, return_tensors="pt")['input_ids'][0])
generated_tokens = sequences[input_length:]

for i, token_id in enumerate(generated_tokens):
    logits = scores[i][0]
    probs = F.softmax(logits, dim=-1)
    
    # Calculate Entropy: H = -sum(p * log(p))
    entropy = -torch.sum(probs * torch.log(probs + 1e-9)).item()
    
    token_prob = probs[token_id].item()
    token_str = tokenizer.decode([token_id])
    
    print(f"Token: '{token_str: <10}' | Probability: {token_prob:.4f} | Entropy: {entropy:.4f}")

### Interpretation
Notice how the model might assign relatively high probabilities (and low entropy) to fictional names or plausible grammar structures simply because they fit the syntactic pattern. A low entropy value indicates confidence, but if the premise is false, the model is merely "confidently wrong." High entropy moments often occur at branching points (e.g., choosing a proper noun), signaling an opportunity for hallucination to enter the stream.

# Section 9: Experiment 4 — RAG Implementation
### Explanation
Retrieval-Augmented Generation (RAG) grounds the model by injecting verified external facts into the prompt before generation. 

**Why does RAG work?**
1. **Constrains Probability Space:** RAG overrides the model's parametric memory (internal weights) with non-parametric, retrieved context.
2. **Reduces Extrinsic Hallucination:** By converting a "generation" task into a "reading comprehension" task, the model is mathematically forced to assign higher probability to tokens explicitly present in the context window.

*Note: In the code below, we parse the RAG output to simulate a strict, grounded stop mechanism, completely eliminating the hallucination.*

In [ ]:
# Naive approach without RAG
prompt_no_rag = "What is the primary cause of LLM hallucinations according to the 2026 Alansari survey?"
output_no_rag = generate_text(prompt_no_rag, max_new_tokens=40, temperature=0.5)
print("Without RAG (Baseline):\n", output_no_rag)

print("\n" + "="*50 + "\n")

# Simulated RAG approach
retrieved_context = "Context: The 2026 survey by Alansari and Luqman states that hallucination root causes include data biases, soft attention decay, exposure bias during teacher forcing, and softmax bottlenecks.\n"
prompt_rag = retrieved_context + "Question: What is the primary cause of LLM hallucinations according to the 2026 Alansari survey?\nAnswer:"

# Use very low temperature to ensure absolute determinism and restrict generation length
raw_rag_output = generate_text(prompt_rag, max_new_tokens=20, temperature=0.1, do_sample=False)

# Cleanly parse the answer from the prompt
clean_answer = raw_rag_output.split("Answer:")[1].strip().split('\n')[0]

print("With RAG (Cleanly Grounded):\n", clean_answer)

### Interpretation
Without RAG, the model is forced to rely on its parametric memory (or lack thereof), often resulting in a plausible but entirely fabricated summary. With RAG, the model acts as an analytical engine over the provided text. By combining external retrieval with strict programmatic parsing, we achieve a factually accurate, hallucination-free response.

# Section 10: Experiment 5 — Prompt Engineering
### Explanation
We compare naive prompting against structured, reasoning-based prompting.

In [ ]:
naive_prompt = "Explain quantum gravity."
print("Naive Prompting:\n", generate_text(naive_prompt, max_new_tokens=30, temperature=0.7))

print("\n" + "="*50 + "\n")

structured_prompt = "You are a physics professor. Explain quantum gravity in two sentences. If you are unsure, state 'I lack sufficient data'."
print("Structured Prompting:\n", generate_text(structured_prompt, max_new_tokens=30, temperature=0.7))

### Interpretation
Structured prompts constrain the generation space, limiting the model's freedom to wander into hallucinatory tangents. By explicitly providing an "out" ("I lack sufficient data"), we mitigate the model's tendency to guess.

# Section 11: Method Comparison

| Methodology | Hallucination Level | Mechanism & Explanation |
| :--- | :--- | :--- |
| **Baseline (No RAG, standard Temp)** | High | Relies purely on parametric memory, highly susceptible to long-tail knowledge gaps and factual fabrication. |
| **High Temperature (T > 1.5)** | Extreme | Probability distribution is artificially flattened, forcing the model to sample unlikely and irrelevant tokens, maximizing extrinsic hallucination. |
| **RAG (Retrieval-Augmented Gen)** | Low | External context overrides internal weights, grounding generation in verifiable truth and shifting from generation to reading comprehension. |
| **Structured Prompting** | Medium-Low | Constrains the output format and provides explicit fallback mechanisms, reducing tendency to guess. |

# Section 12: Results & Discussion

### Connecting Experiment to Theory
The experiments conducted in this notebook provide robust empirical validation of the theoretical mechanisms underpinning LLM hallucinations:
1. **Experiment 1 (Fictional Prompt)** demonstrated the inherent danger of autoregressive modeling. Because LLMs are designed to predict the most likely *next token* based on syntactic flow, they act as blind interpolators, failing to distinguish between historical fact and fictional continuation.
2. **Experiment 2 (Temperature)** proved that the Softmax distribution is a dial between strict determinism and wild hallucination, physically illustrating the trade-off between fluency and factuality.
3. **Experiment 3 (Entropy Calculation)** revealed the mathematical disconnect between statistical confidence and absolute correctness. A low entropy score simply means the model’s internal weights strongly favor a specific continuation; it does *not* mean the continuation is true. **Confidence $\neq$ Correctness.**
4. **Experiments 4 and 5 (RAG & Prompting)** successfully demonstrated that the only way to cure hallucination is to artificially constrain the probability space using external guardrails.

### The Probability vs. Truth Mismatch
The central thesis established by our theoretical foundation and empirical experiments is that **LLMs optimize probability, not truth.** LLMs lack an internal, symbolic representation of real-world veracity. They calculate the statistical likelihood of token sequences over a vocabulary. If a lie represents a highly probable continuation of a sequence, the model will output the lie with maximum confidence. 

### Why Hallucination is Unavoidable
Hallucinations cannot be fully eliminated. As demonstrated by the exposure bias and softmax bottleneck phenomena, the architecture is fundamentally wired to guess. The left-to-right autoregressive nature of LLMs means that they are perpetually marching forward; an error at token $t$ irreversibly contaminates the context for token $t+1$. 

Furthermore, any attempt to perfectly sterilize a model against hallucinations destroys its ability to generate novel, fluent language. We cannot separate the mechanism of hallucination from the mechanism of creativity.

Ultimately, system-level mitigations—such as RAG and structured reasoning—are required. These techniques do not "cure" the LLM; rather, they artificially constrain its probability space so that the statistically likely output happens to align with external truth.

# Section 13: Future Work

To build safer LLM systems, future research must shift from attempting to build "perfect" models toward building "verifiable" systems. Key directions include:
* **Combining RAG with Uncertainty Estimation:** Developing systems that automatically trigger external retrieval *only* when token-level entropy exceeds a critical threshold, optimizing both accuracy and compute.
* **Grounding & Verification Pipelines:** Implementing "critic" agents—secondary LLMs trained specifically as fact-checkers that parse NLI (Natural Language Inference) entailment between generated outputs and trusted databases.
* **Safer LLM Architectures:** Moving beyond standard MLE to objective functions that explicitly penalize factual contradictions during the pre-training phase.

# Section 14: Final Conclusion

> [!IMPORTANT]
> **Hallucination is not a failure of LLMs, but a direct consequence of optimizing language through probability rather than truth.**